# 09b - Agentic Verification Loop (item 7)

(Numbered alongside `09_dpo_training.ipynb` -- both are item 9/7 respectively; run in either order, they don't depend on each other.)

Wraps the rule-based verifier, the counterfactual direction-reversal probe, the learned verifier, and conformal calibration into ONE decision loop (`agents.py`'s `AgenticVerifier`) with real conditional branching: cheap checks run first, expensive tools (learned model, calibration) only run when the cheap check was actually uncertain, and an ambiguous case becomes an explicit abstention rather than a forced guess. Read `agents.py`'s module docstring for what specifically makes this "agentic" rather than a labeled pipeline, and for the honest scope note on retrofitting this pattern into the existing LLM-based recommendation pipeline (not attempted here).

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
import json
import numpy as np
from graph_utils import CitationGraph, load_relational_claims, verify
from hard_negatives import build_tfidf_index
from verifier_features import build_dataset
import verifier_model as vm
from calibration import split_indices, calibrate_threshold, nonconformity_scores
from agents import AgenticVerifier, evaluate_agent, select_showcase_examples, render_showcase_latex
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

graph = CitationGraph.load(REFS_CACHE)
all_claims = load_relational_claims(BENCH, CITE_RESULTS)

paper_abstracts = {}
with open(BENCH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for gt in rec.get("ground_truth_papers") or []:
            pid, ab = gt.get("paper_id"), gt.get("abstract")
            if pid and ab:
                paper_abstracts[pid] = ab
tfidf_vectors = build_tfidf_index(paper_abstracts)

silver = vm.load_silver_labels(CITE_RESULTS)
gold = vm.load_gold_labels(BENCH_VERIFIED) if BENCH_VERIFIED else {}
silver_clean = {k: v for k, v in silver.items() if k not in gold}

X_silver, y_silver, meta_silver = build_dataset(all_claims, graph, tfidf_vectors, silver_clean)
X_gold, y_gold, meta_gold = build_dataset(all_claims, graph, tfidf_vectors, gold)
print(f"Silver: {X_silver.shape} | Gold: {X_gold.shape}")

train_idx, calib_idx = split_indices(len(y_silver), frac=0.85, seed=0)
X_tr, y_tr = X_silver[train_idx], y_silver[train_idx]
X_cal, y_cal = X_silver[calib_idx], y_silver[calib_idx]

scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=2000, class_weight="balanced").fit(scaler.transform(X_tr), y_tr)

probs_cal = model.predict_proba(scaler.transform(X_cal))
scores_cal = nonconformity_scores(probs_cal, y_cal)
qhat = calibrate_threshold(scores_cal, alpha=0.1)
print(f"Conformal threshold (alpha=0.1): qhat={qhat:.4f}")

agent = AgenticVerifier(graph, tfidf_vectors, learned_model=model, learned_scaler=scaler, conformal_qhat=qhat)

print("\nRunning agentic verifier over gold claims...")
agent_results = evaluate_agent(agent, meta_gold, y_gold)
print(f"n={agent_results['n']} escalation_rate={agent_results['escalation_rate']:.3f} "
      f"fragile_rate={agent_results['fragile_rate']:.3f} abstention_rate={agent_results['abstention_rate']:.3f}")
if agent_results["metrics_on_answered"]:
    m = agent_results["metrics_on_answered"]
    print(f"On answered claims (n={agent_results['n_answered']}): "
          f"precision={m['precision']:.3f} recall={m['recall']:.3f} f1={m['f1']:.3f} "
          f"(F1 CI [{m['f1_ci'][0]:.3f}, {m['f1_ci'][1]:.3f}])")

rule_preds = np.array([1 if verify(m["paper_a"], m["relation"], m["paper_b"], graph) == "supported" else 0
                       for m in meta_gold])
rule_metrics = vm.bootstrap_prf1(y_gold, rule_preds)
print(f"\nFor comparison, plain rule (no agentic loop, answers every claim): "
      f"precision={rule_metrics['precision']:.3f} recall={rule_metrics['recall']:.3f} f1={rule_metrics['f1']:.3f}")

print("\nSample reasoning traces:")
for t in agent_results["sample_traces"]:
    print(f"\n  {t['paper_a']} --{t['relation']}--> {t['paper_b']}  =>  {t['verdict']}")
    for line in t["trace"]:
        print("    -", line)

with open(OUT_DIR / "agentic_verifier_results.json", "w", encoding="utf-8") as f:
    json.dump({"agent": agent_results, "plain_rule_baseline": rule_metrics}, f, indent=2)
print("\nSaved to", OUT_DIR / "agentic_verifier_results.json")


## Qualitative case study for the manuscript

Selects one real example per distinct (escalated, fragile, verdict) behavior combination actually observed on gold, and renders it as a ready-to-paste LaTeX fragment (table + full reasoning traces) for the manuscript's "Additional Qualitative Examples" appendix. See `agents.py`'s `select_showcase_examples`/`render_showcase_latex` docstrings -- nothing here is fabricated or cherry-picked beyond "first real example of each bucket that exists in gold"; buckets with zero real examples are simply absent, not filled in.

In [ ]:
showcase = select_showcase_examples(agent, meta_gold, y_gold, max_total=8)
print(f"Selected {len(showcase)} showcase examples covering these behavior buckets:")
for ex in showcase:
    print(f"  escalated={ex['escalated']!s:5} fragile={ex['fragile']!s:5} "
          f"verdict={ex['verdict']:12s} gold={ex['gold_label']:12s} relation={ex['relation']}")

latex_fragment = render_showcase_latex(showcase, n_full_traces=2)
print("\n--- LaTeX fragment preview ---\n")
print(latex_fragment[:1500])

with open(OUT_DIR / "agentic_showcase.json", "w", encoding="utf-8") as f:
    json.dump(showcase, f, indent=2)
with open(OUT_DIR / "agentic_showcase.tex", "w", encoding="utf-8") as f:
    f.write(latex_fragment)
print("\nSaved to", OUT_DIR / "agentic_showcase.json", "and", OUT_DIR / "agentic_showcase.tex")
print("Paste agentic_showcase.tex into the manuscript's Additional Qualitative "
      "Examples appendix in place of the \\TODO{} placeholder.")
